# Quantitative XAI Evaluation for AlphaDent YOLO26 (v4)

v4 fixes from the v3 run: correct parsing of decoded e2e rows `(300, 38)=[box4,conf,cls,coeff32]`;
gradient CAMs backprop through a train-mode forward (decoded output is detached); per-image GPU
cache flush stops the OOM cascade; guards for empty agreement frames; `np.trapezoid`.

Discovers every `best.pt`, runs deletion/insertion faithfulness + inter-method agreement per config,
emits per-config CSVs, pooled figures, LaTeX rows, one zip.

In [ ]:
# Clone repo if not running inside it already
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # reduce CUDA fragmentation
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python scipy pandas


In [ ]:
import zipfile
import shutil
import os

input_dir = "/kaggle/input/competitions/alpha-dent/AlphaDent"
working_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

# Check if Kaggle input contains the dataset
if os.path.exists(os.path.join(input_dir, 'images')) and os.listdir(os.path.join(input_dir, 'images')):
    print("Dataset found in Kaggle input. Copying dataset...")
    shutil.copytree(input_dir, working_dir, dirs_exist_ok=True)
    print("Dataset copied successfully. Using input dataset.")
else:
    # Fallback to downloading from Zenodo
    images_dir = os.path.join(working_dir, 'images')
    if not os.path.exists(images_dir) or not os.listdir(images_dir):
        print("Downloading AlphaDent dataset from Zenodo...")
        !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}

        print("Extracting dataset...")
        os.makedirs(working_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(working_dir)

        # Remove zip to save disk space
        if os.path.exists(zip_path):
            os.remove(zip_path)

        # Check if nested folder exists after extraction and move files up if needed
        nested_dir = os.path.join(working_dir, 'AlphaDent')
        if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
            print("Adjusting dataset directory structure...")
            for item in os.listdir(nested_dir):
                shutil.move(os.path.join(nested_dir, item), working_dir)
            os.rmdir(nested_dir)

        print("Dataset prepared successfully in working directory.")
    else:
        print("Dataset already exists in working directory.")

# NOTE: no 4-class label conversion here — this eval needs the ORIGINAL 9-class labels.
# The runner remaps ground truth (cls>=3 -> 3) on the fly when evaluating nc==4 models,
# so a single unconverted label set serves both model families. Converting in place
# would silently break TP/FP scoring for all 9-class configs.


In [ ]:
import os, re, glob, torch

# ---- Config: discover ALL trained weights ----
CANDIDATE_GLOBS = [
    'runs/segment/*/train/weights/best.pt',
    'runs_4classes/segment/*/train/weights/best.pt',
    '/kaggle/working/AlphaDentYolov26/runs*/segment/*/train/weights/best.pt',
    '/kaggle/input/*/**/best.pt',
]
found = []
for g in CANDIDATE_GLOBS:
    found += glob.glob(g, recursive=True)
found = sorted(set(found))

def cfg_name(path):
    m = re.search(r'yolo_seg_(yolo26[nsmlx])_proj_(\d+)', path)
    if m:
        tag = '4class_' if ('4class' in path.lower()) else ''
        return f"{tag}{m.group(1)}_{m.group(2)}", int(m.group(2))
    return os.path.basename(os.path.dirname(os.path.dirname(path))), 640

CONFIGS = []   # (name, weights_path, imgsz)
seen = set()
for p in found:
    name, sz = cfg_name(p)
    if name not in seen:
        seen.add(name)
        CONFIGS.append((name, p, sz))
# Pipeline filter: 9-class only
CONFIGS = [c for c in CONFIGS if ('4class' in c[0].lower() or '4class' in c[1].lower()) == False]
assert CONFIGS, 'No 9-class best.pt found (4-class weight paths must contain \'4class\').'

DATA_DIR = './AlphaDent'
N_IMAGES = 30
N_STEPS  = 20
OUT_DIR  = 'xai_eval'
DEVICE   = 0 if torch.cuda.is_available() else 'cpu'
METHODS  = ['EigenCAM', 'GradCAM', 'GradCAM++', 'XGradCAM', 'HiResCAM', 'LayerCAM', 'EigenGradCAM']

os.makedirs(OUT_DIR, exist_ok=True)
print(f'{len(CONFIGS)} configs discovered:')
for n, p, sz in CONFIGS:
    print(f'  {n:24s} imgsz={sz}  {p}')


In [ ]:
import os, cv2, numpy as np, torch

def letterbox(img, size, pad_val=114):
    h, w = img.shape[:2]
    r = size / max(h, w)
    nh, nw = int(h * r), int(w * r)
    rs = cv2.resize(img, (nw, nh))
    canvas = np.full((size, size, 3), pad_val, dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = rs
    return canvas, r, (top, left)

def to_tensor(img_bgr, device):
    x = img_bgr.astype(np.float32) / 255.0
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(device)

def unwrap_tensor(out):
    """Descend nested list/tuple output until first Tensor."""
    while isinstance(out, (list, tuple)):
        out = out[0]
    return out

def parse_yolo_preds(t, nc):
    """Normalize YOLO output. Returns (kind, mat):
       kind='raw': mat (C,N) channel-first grids, channels = [box4, cls nc, coeff...]
       kind='e2e': mat (N,K) decoded rows = [x1,y1,x2,y2,conf,cls,(coeff...)]"""
    if t.ndim == 3:
        t = t[0]
    if t.ndim != 2:
        raise ValueError(f'unexpected pred shape {tuple(t.shape)}')
    r, c = t.shape
    C_raw = 4 + nc + 32          # channel-first grid width (box+cls+mask coeffs)
    if r == C_raw or r == 4 + nc:
        return 'raw', t
    if c == C_raw or c == 4 + nc:
        return 'raw', t.T
    if c in (6, 38):              # decoded rows: 6 cols, or 6+32 with mask coeffs
        return 'e2e', t
    if r in (6, 38):
        return 'e2e', t.T
    # fallback heuristics
    if r < c and r >= 4 + nc:
        return 'raw', t
    if c < r and c >= 4 + nc:
        return 'raw', t.T
    raise ValueError(f'cannot parse pred shape {tuple(t.shape)} with nc={nc}')

def pred_class_and_score(t, nc, class_idx=None):
    """Differentiable: returns (class_idx, scalar score tensor for that class)."""
    kind, mat = parse_yolo_preds(t, nc)
    if kind == 'raw':
        if class_idx is None:
            class_idx = int(mat[4:4 + nc, :].sum(dim=-1).argmax())
        return class_idx, mat[4 + class_idx, :].max()
    det = mat                      # rows: [x1,y1,x2,y2,conf,cls,...]
    if class_idx is None:
        class_idx = int(det[det[:, 4].argmax(), 5])
    m = det[:, 5].long() == class_idx
    score = det[m, 4].max() if m.any() else det[:, 4].sum() * 0.0
    return class_idx, score

TRAPZ = getattr(np, 'trapezoid', None) or np.trapz

def collect_cls_logit_maps(out, nc, acc=None):
    """Recursively collect 4-D head tensors from a TRAIN-mode forward whose channel
    count can contain nc class logits at the tail (excludes 32-ch mask-coeff maps)."""
    if acc is None:
        acc = []
    if torch.is_tensor(out):
        if out.ndim == 4 and out.shape[1] > max(nc, 32):
            acc.append(out)
    elif isinstance(out, (list, tuple)):
        for o in out:
            collect_cls_logit_maps(o, nc, acc)
    return acc

@torch.no_grad()
def class_score(yolo, img_bgr, class_idx, device, nc):
    out = unwrap_tensor(yolo.model(to_tensor(img_bgr, device)))
    _, s = pred_class_and_score(out, nc, class_idx)
    return float(s.cpu())

def poly_txt_to_boxes(txt_path, img_w, img_h):
    boxes = []
    if not os.path.exists(txt_path):
        return boxes
    for line in open(txt_path):
        p = line.split()
        if len(p) < 7:
            continue
        cls = int(p[0])
        xs = np.array(p[1::2], dtype=float) * img_w
        ys = np.array(p[2::2], dtype=float) * img_h
        boxes.append((cls, xs.min(), ys.min(), xs.max(), ys.max()))
    return boxes

def box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

def _flatten_tensors(out, acc=None):
    if acc is None:
        acc = []
    if torch.is_tensor(out):
        acc.append(out)
    elif isinstance(out, (list, tuple)):
        for o in out:
            _flatten_tensors(o, acc)
    return acc

class YOLOExplainer:
    """
    Custom Class Activation Mapping (CAM) generator for Ultralytics YOLO models.
    Supports Eigen-CAM, Grad-CAM, Grad-CAM++, XGrad-CAM, HiRes-CAM, Layer-CAM, and EigenGrad-CAM.
    """
    def __init__(self, model, target_layer, method='eigencam'):
        self.model = model
        self.target_layer = target_layer
        self.method = method.lower()
        self.activations = None
        self.gradients = None
        
        # Register hooks
        self.forward_hook = target_layer.register_forward_hook(self._forward_hook_fn)
        if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
            self.backward_hook = target_layer.register_full_backward_hook(self._backward_hook_fn)
            
    def _forward_hook_fn(self, module, input, output):
        if isinstance(output, (list, tuple)):
            self.activations = output[0].clone()
        else:
            self.activations = output.clone()
            
    def _backward_hook_fn(self, module, grad_input, grad_output):
        if isinstance(grad_output, (list, tuple)):
            self.gradients = grad_output[0].clone()
        else:
            self.gradients = grad_output.clone()
            
    def remove_hooks(self):
        self.forward_hook.remove()
        if hasattr(self, 'backward_hook'):
            self.backward_hook.remove()
            
    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        
        with torch.set_grad_enabled(True):
            if self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                input_tensor.requires_grad = True
                
            # Ensure model weights are on the same device as input_tensor
            self.model.model.to(input_tensor.device)
            
            # Run raw PyTorch forward pass to track gradients and bypass Results wrapper
            outputs = self.model.model(input_tensor)
            
            preds = unwrap_tensor(outputs)
                
            if self.method == 'eigencam':
                act = self.activations.detach().cpu().numpy()[0]
                C, H, W = act.shape
                matrix = act.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap
                
            elif self.method in ['gradcam', 'gradcam++', 'xgradcam', 'hirescam', 'layercam', 'eigengradcam']:
                nc = getattr(self.model.model, 'nc', None) or len(self.model.names)
                if class_idx is None:
                    class_idx, _ = pred_class_and_score(preds, nc, None)
                # Decoded e2e output is detached from the graph; re-run forward in
                # train mode to get raw differentiable head logits (BN kept in eval).
                net = self.model.model
                net.train()
                for _mod in net.modules():
                    if isinstance(_mod, nn.modules.batchnorm._BatchNorm):
                        _mod.eval()
                try:
                    raw_out = net(input_tensor)
                    maps = collect_cls_logit_maps(raw_out, nc)
                    if not maps:
                        raise RuntimeError(f'no class-logit maps in train-mode output; shapes='
                                           + str([tuple(t.shape) for t in _flatten_tensors(raw_out)]))
                    score = sum(m[:, -nc + class_idx, :, :].sigmoid().sum() for m in maps)
                    score.backward()
                finally:
                    net.eval()
            
            act = self.activations.detach().cpu().numpy()[0]
            grad = self.gradients.detach().cpu().numpy()[0]
            
            if self.method == 'gradcam':
                weights = np.mean(grad, axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap
                
            elif self.method == 'gradcam++':
                grads_power_2 = grad ** 2
                grads_power_3 = grad ** 3
                sum_act = np.sum(act, axis=(1, 2))
                alpha = grads_power_2 / (2 * grads_power_2 + sum_act[:, None, None] * grads_power_3 + 1e-8)
                weights = np.sum(alpha * np.maximum(grad, 0), axis=(1, 2))
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'xgradcam':
                sum_act = np.sum(act, axis=(1, 2))
                weights = np.sum(grad * act, axis=(1, 2)) / (sum_act + 1e-8)
                cam = np.zeros(act.shape[1:], dtype=np.float32)
                for i, w in enumerate(weights):
                    cam += w * act[i, :, :]
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'hirescam':
                cam = np.sum(act * grad, axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'layercam':
                cam = np.sum(act * np.maximum(grad, 0), axis=0)
                cam = np.maximum(cam, 0)
                heatmap = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                return heatmap

            elif self.method == 'eigengradcam':
                act_grad = act * np.maximum(grad, 0)
                C, H, W = act_grad.shape
                matrix = act_grad.reshape(C, H * W).T
                matrix = matrix - np.mean(matrix, axis=0)
                U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
                projection = U[:, 0].reshape(H, W)
                heatmap = (projection - projection.min()) / (projection.max() - projection.min() + 1e-8)
                return heatmap

def show_heatmap(img_path, heatmap, title='Attention Map', save_path=None):
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, superimposed_img)
        
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')
    axes[0].set_title('Original Image')
    
    axes[1].imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    axes[1].axis('off')
    axes[1].set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Run EVERYTHING: faithfulness + agreement for every discovered config ----
import pandas as pd
from itertools import combinations
from scipy.stats import spearmanr, mannwhitneyu
from ultralytics import YOLO

# Defensive cleanup: drop stale globals from earlier cell runs in this kernel
import gc as _gc
for _g in ['model', 'heat_store', '_raw', '_t', 'fdf', 'adf']:
    if _g in dir():
        del globals()[_g]
_gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'GPU free before start: {free_b/1e9:.1f} / {total_b/1e9:.1f} GB')
    assert free_b > 4e9, 'GPU mostly occupied — RESTART THE KERNEL and Run All.'

val_images = sorted(glob.glob(f'{DATA_DIR}/images/valid/*.jpg') + glob.glob(f'{DATA_DIR}/images/valid/*.png'))[:N_IMAGES]
print(f'{len(val_images)} eval images')

def run_faithfulness(model, nc, target_layer, imgsz, out_sub):
    rows, heat_store = [], {}
    for ip in val_images:
        img = cv2.imread(ip)
        img_lb, _, _ = letterbox(img, imgsz)
        try:
            with torch.no_grad():
                out = unwrap_tensor(model.model(to_tensor(img_lb, DEVICE)))
                cls, _ = pred_class_and_score(out, nc, None)
        except Exception as e:
            print('  skip', os.path.basename(ip), repr(e))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        base = class_score(model, img_lb, cls, DEVICE, nc) + 1e-8
        blur = cv2.GaussianBlur(img_lb, (51, 51), 0)
        h, w = img_lb.shape[:2]
        flat_img, flat_blur = img_lb.reshape(-1, 3), blur.reshape(-1, 3)
        for m in METHODS:
            ex = YOLOExplainer(model, target_layer, method=m)
            try:
                heat = ex.generate(to_tensor(img_lb, DEVICE), class_idx=cls)
                ex.remove_hooks()
                hm = cv2.resize(heat, (w, h)).flatten()
                order = np.argsort(-hm)
                dele = [1.0]
                ins = [class_score(model, blur, cls, DEVICE, nc) / base]
                for k in range(1, N_STEPS + 1):
                    idx = order[: int(len(order) * k / N_STEPS)]
                    d = flat_img.copy(); d[idx] = 114
                    i = flat_blur.copy(); i[idx] = flat_img[idx]
                    dele.append(class_score(model, d.reshape(h, w, 3), cls, DEVICE, nc) / base)
                    ins.append(class_score(model, i.reshape(h, w, 3), cls, DEVICE, nc) / base)
                rows.append({'image': os.path.basename(ip), 'method': m, 'class': cls,
                             'deletion_auc': float(TRAPZ(dele, dx=1/N_STEPS)),
                             'insertion_auc': float(TRAPZ(ins, dx=1/N_STEPS))})
                heat_store[(ip, m)] = heat
            except Exception as e:
                ex.remove_hooks(); print('  fail', m, os.path.basename(ip), repr(e))
            finally:
                model.model.zero_grad(set_to_none=True)
        import gc as _gc; _gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if rows:
            pd.DataFrame(rows).to_csv(f'{out_sub}/faithfulness_raw.csv', index=False)
    return pd.DataFrame(rows), heat_store

def run_agreement(model, nc, imgsz, heat_store, out_sub):
    rows = []
    for ip in val_images:
        hs = [cv2.resize(heat_store[(ip, m)], (64, 64)).flatten() for m in METHODS if (ip, m) in heat_store]
        if len(hs) < 2:
            continue
        ag = float(np.nanmean([spearmanr(a, b).correlation for a, b in combinations(hs, 2)]))
        img = cv2.imread(ip); H, W = img.shape[:2]
        r = model.predict(ip, imgsz=imgsz, conf=0.25, verbose=False, device=DEVICE)[0]
        correct, conf = 0, 0.0
        if len(r.boxes):
            b = r.boxes[r.boxes.conf.argmax()]
            conf = float(b.conf)
            pcls = int(b.cls); pbox = b.xyxy[0].tolist()
            gts = poly_txt_to_boxes(f"{DATA_DIR}/labels/valid/{os.path.splitext(os.path.basename(ip))[0]}.txt", W, H)
            if nc == 4:  # model trained on merged labels; remap 9-class GT
                gts = [(min(g[0], 3), *g[1:]) for g in gts]
            correct = int(any(g[0] == pcls and box_iou(pbox, g[1:]) >= 0.5 for g in gts))
        rows.append({'image': os.path.basename(ip), 'agreement': ag, 'top_conf': conf, 'correct': correct})
    df = pd.DataFrame(rows)
    df.to_csv(f'{out_sub}/agreement.csv', index=False)
    return df

all_faith, all_agree = [], []
for name, wpath, imgsz in CONFIGS:
    print(f'\n===== {name} (imgsz={imgsz}) =====')
    out_sub = f'{OUT_DIR}/{name}'; os.makedirs(out_sub, exist_ok=True)
    model = YOLO(wpath); model.model.to(DEVICE).eval()
    nc = getattr(model.model, 'nc', None) or len(model.names)
    target_layer = model.model.model[22]
    # diagnostic once per config
    _lb, _, _ = letterbox(cv2.imread(val_images[0]), imgsz)
    with torch.no_grad():
        _t = unwrap_tensor(model.model(to_tensor(_lb, DEVICE)))
    print(f'  nc={nc} raw shape={tuple(_t.shape)} parsed={parse_yolo_preds(_t, nc)[0]}')
    fdf, heat_store = run_faithfulness(model, nc, target_layer, imgsz, out_sub)
    if fdf.empty:
        print('  no faithfulness rows, skipping agreement'); continue
    fdf['config'] = name; all_faith.append(fdf)
    adf = run_agreement(model, nc, imgsz, heat_store, out_sub)
    if adf.empty:
        print('  no agreement rows (need >=2 methods per image)'); continue
    adf['config'] = name; all_agree.append(adf)
    tp, fp = adf[adf.correct == 1].agreement, adf[adf.correct == 0].agreement
    msg = f'  agreement: TP={tp.mean():.3f}(n={len(tp)}) FP={fp.mean():.3f}(n={len(fp)})'
    if len(tp) > 2 and len(fp) > 2:
        u, pv = mannwhitneyu(tp, fp, alternative='greater')
        msg += f'  MWU p={pv:.4f}'
    print(msg)
    del model, heat_store
    if torch.cuda.is_available(): torch.cuda.empty_cache()

assert all_faith, 'Nothing produced.'
faith = pd.concat(all_faith)
agree = pd.concat(all_agree) if all_agree else pd.DataFrame(columns=['image','agreement','top_conf','correct','config'])
faith.to_csv(f'{OUT_DIR}/faithfulness_all.csv', index=False)
agree.to_csv(f'{OUT_DIR}/agreement_all.csv', index=False)
print('\nCombined summary (mean deletion/insertion AUC per config x method):')
print(faith.groupby(['config','method'])[['deletion_auc','insertion_auc']].mean().round(4))


In [ ]:
# ---- Aggregate outputs: figures, LaTeX rows, stats, zip ----
import pandas as pd, matplotlib.pyplot as plt, shutil
from scipy.stats import spearmanr, mannwhitneyu

faith = pd.read_csv(f'{OUT_DIR}/faithfulness_all.csv')
agree = pd.read_csv(f'{OUT_DIR}/agreement_all.csv')

# Method-level summary pooled across configs
pooled = faith.groupby('method')[['deletion_auc','insertion_auc']].agg(['mean','std']).round(4)
pooled.to_csv(f'{OUT_DIR}/faithfulness_pooled.csv')
print(pooled)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
faith.boxplot(column='deletion_auc', by='method', ax=axes[0], rot=45)
axes[0].set_title('Deletion AUC (lower = faithful)'); axes[0].set_xlabel('')
faith.boxplot(column='insertion_auc', by='method', ax=axes[1], rot=45)
axes[1].set_title('Insertion AUC (higher = faithful)'); axes[1].set_xlabel('')
plt.suptitle(''); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_faithfulness.png', dpi=200); plt.show()

plt.figure(figsize=(5.5, 4.5))
for c, lbl, col in [(1, 'correct (TP)', 'tab:green'), (0, 'incorrect/miss', 'tab:red')]:
    sub = agree[agree.correct == c]
    plt.scatter(sub.agreement, sub.top_conf, label=lbl, c=col, alpha=0.6, s=18)
plt.xlabel('inter-method agreement (mean pairwise Spearman)')
plt.ylabel('top prediction confidence'); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_agreement.png', dpi=200); plt.show()

tp, fp = agree[agree.correct == 1].agreement, agree[agree.correct == 0].agreement
print(f'\nPooled agreement: TP={tp.mean():.3f}(n={len(tp)}) FP={fp.mean():.3f}(n={len(fp)})')
if len(tp) > 2 and len(fp) > 2:
    u, pv = mannwhitneyu(tp, fp, alternative='greater')
    print(f'Mann-Whitney U (TP > FP): U={u:.0f}, p={pv:.5f}')
print(f'Spearman(agreement, conf) = {spearmanr(agree.agreement, agree.top_conf).correlation:.3f}')

print('\n% --- LaTeX rows: pooled faithfulness (method & del AUC & ins AUC) ---')
for m in pooled.index:
    print(f"{m} & {pooled.loc[m,('deletion_auc','mean')]:.3f} $\\pm$ {pooled.loc[m,('deletion_auc','std')]:.3f} & "
          f"{pooled.loc[m,('insertion_auc','mean')]:.3f} $\\pm$ {pooled.loc[m,('insertion_auc','std')]:.3f} \\\\")

dst = '/kaggle/working/xai_eval_results' if os.path.exists('/kaggle/working') else 'xai_eval_results'
shutil.make_archive(dst, 'zip', OUT_DIR)
print('\nzipped ->', dst + '.zip')
